In [1]:
import geopandas as gpd
import pandas as pd

In [2]:
gdf = gpd.read_file("Data/kd_z_nevarnost_enriched_verified.geojson")

In [3]:
gdf.columns

Index(['ESD', 'EID', 'IME', 'SINONIMI', 'OPIS', 'ZVRST', 'TIP', 'GESLA',
       'DATACIJA', 'LOKACIJAOPIS', 'OBCINA', 'ZAVOD', 'USMERITVE', 'STATUS',
       'PREDPIS', 'PREDPISHTTP', 'VELJAVNOST', 'FOTOAVTOR', 'FOTODATOTEKA',
       'POVEZAVA1', 'SPOMENIK', 'OBCINAID', 'QR', 'X', 'Y', 'PHOTOURL',
       'poplave_ocena', 'poplave', 'pozar', 'plazovi', 'regija', 'UE_UIME',
       'potres', 'predominant_material', 'material_confidence',
       'fire_danger_revised', 'flood_danger_revised',
       'earthquake_danger_revised', 'landslide_danger_revised',
       'danger_revision_reasoning', 'verification_status',
       'verification_notes', 'sources', 'research_confidence', 'geometry'],
      dtype='str')

In [4]:
embed_cols = gdf[['IME', 'SINONIMI', 'OPIS', 'ZVRST', 'TIP', 'GESLA', 'DATACIJA', 'LOKACIJAOPIS', 'predominant_material', 'UE_UIME', 'OBCINA']]
meta_data_cols = gdf[['EID', 'OBCINA', 'STATUS', 'SPOMENIK', 'UE_UIME', 'predominant_material', 'fire_danger_revised', 'flood_danger_revised',
       'earthquake_danger_revised', 'landslide_danger_revised']]

In [15]:
texts = embed.apply(lambda row: " | ".join(
    str(v) for v in row if pd.notna(v) and str(v).strip()
),axis=1)

def row_to_text(row):
    return " | ".join(
        str(v) for v in row if pd.notna(v) and str(v).strip()
    )

In [12]:
gdf['embed_text'] = gdf.apply(row_to_text, axis=1)

In [ ]:
zapisi = []

for _, row in gdf.iterrows():
    zapisi.append({
        "eid" : row['EID'],
        "text" : row['embed_text'],
        "meta_data" : {col: row[col] for col in meta_data_cols}
    })

In [ ]:
import os
#from openai import __
import chromadb
#client setup ipd.
EMBED_MODEL='text-embedding-3-small'

In [14]:
chroma_client = chromadb.PersistentClient(path="Data/chroma_db")
collection = chroma_client.get_or_create_collection("kulturna dediscina")

BATCH_SIZE = 100
for i in range(0, len(zapisi), BATCH_SIZE):
    batch = records[i: i+BATCH_SIZE]
    texts = [r['text'] for r in batch]

    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=texts
    )

    collection.add(
        ids=[str(r['eid']) for r in batch],
        documents=[r['text'] for r in batch],
        metadatas=[r['meta_data'] for r in batch]
        embeddings=[item.embedding for item in response.data]
    )

NameError: name 'client' is not defined